# Weight Initialization in PyTorch

This notebook makes every formula from the readings experimentally verifiable. You will:
1. **Apply every `torch.nn.init` function** and verify the resulting statistics
2. **Instrument forward passes** to measure activation variance across a 20-layer network
3. **Demonstrate the symmetry problem** empirically with zero initialization
4. **Compare Xavier vs He** on a 10-layer ReLU MLP trained on MNIST
5. **Verify orthogonal initialization** numerically for an RNN
6. **Audit PyTorch layer defaults** for `Linear`, `Conv2d`, `Embedding`, `LSTM`, and `BatchNorm2d`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

def check_stats(name, t, expected_mean=0.0, expected_var=None, atol_mean=0.05, atol_var=0.1):
    m = t.float().mean().item()
    v = t.float().var().item()
    mean_ok = abs(m - expected_mean) < atol_mean
    var_ok  = True if expected_var is None else abs(v - expected_var) < atol_var
    status = '\u2713' if (mean_ok and var_ok) else '\u2717'
    var_str = f'var={v:.4f} (expected~{expected_var:.4f})' if expected_var is not None else f'var={v:.4f}'
    print(f'{status} {name:<40}  mean={m:.4f}  {var_str}')

## Section 1 — All `torch.nn.init` Functions

We apply every initializer in the `torch.nn.init` module to a `(256, 256)` tensor and verify the resulting mean, variance, and (where applicable) structural properties.

In [ ]:
shape = (256, 256)
t = torch.empty(shape)

# --- Constant initializers ---
print('=== Constant initializers ===')
nn.init.zeros_(t);    check_stats('zeros_',          t, expected_mean=0.0, expected_var=0.0)
nn.init.ones_(t);     check_stats('ones_',           t, expected_mean=1.0, expected_var=0.0)
nn.init.constant_(t, 3.14); check_stats('constant_(3.14)', t, expected_mean=3.14, expected_var=0.0)

# --- Identity / eye ---
print('\n=== Identity / eye ===')
nn.init.eye_(t)
identity_check = torch.allclose(t, torch.eye(256))
print(f"{'\u2713' if identity_check else '\u2717'} eye_  — diagonal=1, off-diagonal=0: {identity_check}")

# --- Random distribution initializers ---
print('\n=== Random distribution initializers ===')
nn.init.uniform_(t, a=-1.0, b=1.0)
# Uniform(-1,1): mean=0, var=(b-a)^2/12 = 4/12 = 0.333
check_stats('uniform_(-1, 1)',     t, expected_mean=0.0, expected_var=1/3)

nn.init.normal_(t, mean=0.0, std=1.0)
check_stats('normal_(0, 1)',       t, expected_mean=0.0, expected_var=1.0)

nn.init.normal_(t, mean=0.0, std=0.01)
check_stats('normal_(0, 0.01)',    t, expected_mean=0.0, expected_var=0.0001)

nn.init.trunc_normal_(t, mean=0.0, std=1.0, a=-2.0, b=2.0)
above_bound = (t.abs() > 2.0).sum().item()
print(f'\u2713 trunc_normal_(-2, 2)  values outside bounds: {above_bound} (should be 0)')
check_stats('trunc_normal_',       t, expected_mean=0.0)

# Sparse: fraction of columns set to zero, non-zeros from N(0, std)
nn.init.sparse_(t, sparsity=0.5, std=0.01)
col_zero_frac = (t == 0).float().mean(dim=0)   # fraction zeros per column
approx_sparsity = col_zero_frac.mean().item()
print(f'\u2713 sparse_(sparsity=0.5)  approx column sparsity: {approx_sparsity:.3f} (expected ~0.5)')

In [ ]:
# --- Xavier / Glorot ---
print('=== Xavier / Glorot ===')
# fan_in = fan_out = 256 for square matrix
fan_in = fan_out = 256

nn.init.xavier_uniform_(t, gain=1.0)
# Var = 2 / (fan_in + fan_out) = 2/512 = 0.00390625
xavier_var = 2.0 / (fan_in + fan_out)
# Uniform(-a, a) where a = gain * sqrt(6 / (fan_in+fan_out))
a = math.sqrt(6.0 / (fan_in + fan_out))
check_stats('xavier_uniform_',     t, expected_mean=0.0, expected_var=xavier_var)
bound_ok = (t.abs() <= a + 1e-6).all().item()
print(f'  Bounds [-{a:.4f}, {a:.4f}] satisfied: {bound_ok}')

nn.init.xavier_normal_(t, gain=1.0)
check_stats('xavier_normal_',      t, expected_mean=0.0, expected_var=xavier_var)

# --- He / Kaiming ---
print('\n=== He / Kaiming ===')
# fan_in mode: Var = 2 / fan_in
he_var_fan_in = 2.0 / fan_in

nn.init.kaiming_uniform_(t, a=0, mode='fan_in', nonlinearity='relu')
check_stats('kaiming_uniform_(fan_in, relu)', t, expected_mean=0.0, expected_var=he_var_fan_in)

nn.init.kaiming_normal_(t, a=0, mode='fan_in', nonlinearity='relu')
check_stats('kaiming_normal_(fan_in, relu)',  t, expected_mean=0.0, expected_var=he_var_fan_in)

# LeakyReLU: gain^2 = 2/(1+a^2), Var = gain^2 / fan_in
a_slope = 0.1
he_var_leaky = 2.0 / ((1 + a_slope**2) * fan_in)
nn.init.kaiming_normal_(t, a=a_slope, mode='fan_in', nonlinearity='leaky_relu')
check_stats('kaiming_normal_(a=0.1, leaky)', t, expected_mean=0.0, expected_var=he_var_leaky)

# fan_out mode: Var = 2 / fan_out
nn.init.kaiming_normal_(t, a=0, mode='fan_out', nonlinearity='relu')
check_stats('kaiming_normal_(fan_out, relu)', t, expected_mean=0.0, expected_var=2.0/fan_out)

# --- Orthogonal ---
print('\n=== Orthogonal ===')
t_sq = torch.empty(256, 256)
nn.init.orthogonal_(t_sq, gain=1.0)
WtW = t_sq.T @ t_sq
diff = (WtW - torch.eye(256)).abs().max().item()
print(f'\u2713 orthogonal_  max|W^T W - I| = {diff:.2e}  (should be ~0)')

# Non-square: (512, 256) — columns are orthonormal
t_tall = torch.empty(512, 256)
nn.init.orthogonal_(t_tall)
WtW_tall = t_tall.T @ t_tall
diff_tall = (WtW_tall - torch.eye(256)).abs().max().item()
print(f'\u2713 orthogonal_ (512,256)  max|W^T W - I| = {diff_tall:.2e}')

# Dirac (Conv)
print('\n=== Dirac (Conv) ===')
conv_w = torch.empty(16, 16, 3, 3)  # (out_ch, in_ch, kH, kW)
nn.init.dirac_(conv_w)
# Center element of each in/out pair should be 1, rest 0
center = conv_w[:16, :16, 1, 1]  # min(in,out) diagonal
center_diag_ok = torch.allclose(center, torch.eye(16))
print(f'\u2713 dirac_  center slice is identity: {center_diag_ok}')

## Section 2 — Variance Propagation Experiment

We build a 20-layer linear network and measure how activation variance evolves through depth under four initialization strategies:
- **zeros** (breaks symmetry; used only for illustration)
- **N(0, 1)** (naive large random)
- **Xavier / Glorot normal** (designed for Tanh/linear activations)
- **He / Kaiming normal** (designed for ReLU)

Each layer is a linear transform with no activation (to isolate the pure linear variance propagation).

In [ ]:
def measure_variance_propagation(init_fn, n_layers=20, width=256, n_samples=1024):
    """Run a random input through n_layers linear layers and record per-layer variance."""
    torch.manual_seed(0)
    x = torch.randn(n_samples, width)
    variances = [x.var().item()]
    for _ in range(n_layers):
        W = torch.empty(width, width)
        init_fn(W)
        b = torch.zeros(width)
        x = x @ W.T + b
        variances.append(x.var().item())
    return variances

inits = {
    'zeros':         lambda W: nn.init.zeros_(W),
    'N(0, 1)':       lambda W: nn.init.normal_(W, 0, 1),
    'Xavier normal': lambda W: nn.init.xavier_normal_(W),
    'He normal':     lambda W: nn.init.kaiming_normal_(W, nonlinearity='relu'),
}

results = {name: measure_variance_propagation(fn) for name, fn in inits.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, vars_ in results.items():
    ax.plot(vars_, 'o-', label=name, linewidth=2, markersize=4)
ax.set_xlabel('Layer'); ax.set_ylabel('Activation variance')
ax.set_title('Variance Propagation (linear scale)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for name, vars_ in results.items():
    safe = [max(v, 1e-30) for v in vars_]
    ax.semilogy(safe, 'o-', label=name, linewidth=2, markersize=4)
ax.set_xlabel('Layer'); ax.set_ylabel('Activation variance (log scale)')
ax.set_title('Variance Propagation (log scale)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('\nFinal-layer variance:')
for name, vars_ in results.items():
    print(f'  {name:<20}: {vars_[-1]:.4e}')

## Section 3 — The Symmetry Problem

When all weights are initialised to zero every neuron in a layer computes the same function and receives the same gradient update. They remain identical throughout training — the network is effectively a single neuron regardless of width.

We demonstrate this by training a 3-layer MLP with all-zero weights and plotting per-neuron weight evolution.

In [ ]:
class MLP(nn.Module):
    def __init__(self, init_fn):
        super().__init__()
        self.fc1 = nn.Linear(10, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
        for layer in [self.fc1, self.fc2, self.fc3]:
            init_fn(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

torch.manual_seed(0)
X = torch.randn(256, 10)
y = X[:, :3].sum(1, keepdim=True)

models = {
    'zeros':          MLP(lambda w: nn.init.zeros_(w)),
    'kaiming_normal': MLP(lambda w: nn.init.kaiming_normal_(w)),
}

weight_histories = {name: [] for name in models}

n_steps = 100
for name, model in models.items():
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    for step in range(n_steps):
        opt.zero_grad()
        F.mse_loss(model(X), y).backward()
        opt.step()
        # Record weights of first 8 neurons in fc1
        weight_histories[name].append(model.fc1.weight[:8, 0].detach().clone())

# Plot per-neuron weights over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, hist) in zip(axes, weight_histories.items()):
    history = torch.stack(hist).numpy()  # (n_steps, 8)
    for neuron in range(8):
        ax.plot(history[:, neuron], linewidth=1.5, alpha=0.8)
    ax.set_title(f'fc1 neuron weights — {name} init')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Weight value (w.r.t. input[0])')
    ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Quantify: max pairwise difference in weights at end
for name, model in models.items():
    w = model.fc1.weight.detach()
    diffs = (w.unsqueeze(0) - w.unsqueeze(1)).abs().max().item()
    print(f'{name:<20}: max pairwise weight diff in fc1 = {diffs:.6f}')

## Section 4 — Xavier vs He on a 10-Layer ReLU MLP (MNIST)

We train two identical 10-layer ReLU MLPs on MNIST — one with Xavier normal init, one with He/Kaiming normal — and compare:
- Training loss curves
- Validation accuracy
- Per-layer activation variance at epoch 1

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_set = torchvision.datasets.MNIST('/tmp/mnist', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)
print(f'MNIST: {len(train_set)} train, {len(test_set)} test')

In [ ]:
class DeepMLP(nn.Module):
    """10 hidden layers of width 256 with ReLU."""
    def __init__(self, init_name='kaiming'):
        super().__init__()
        layers = [nn.Linear(784, 256), nn.ReLU()]
        for _ in range(9):
            layers += [nn.Linear(256, 256), nn.ReLU()]
        layers.append(nn.Linear(256, 10))
        self.net = nn.Sequential(*layers)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init_name == 'xavier':
                    nn.init.xavier_normal_(m.weight)
                else:
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

def collect_layer_variances(model, loader, n_batches=2):
    """Record activation variance at each Linear layer output."""
    model.eval()
    layer_acts = [[] for _ in range(11)]  # 10 hidden + 1 output
    hooks = []
    layer_idx = [0]

    def make_hook(idx):
        def hook(m, inp, out):
            layer_acts[idx].append(out.detach().float())
        return hook

    for i, m in enumerate(model.modules()):
        if isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(make_hook(layer_idx[0])))
            layer_idx[0] += 1

    with torch.no_grad():
        for batch_i, (imgs, _) in enumerate(loader):
            if batch_i >= n_batches: break
            model(imgs.to(device))

    for h in hooks: h.remove()
    return [torch.cat(acts, dim=0).var().item() for acts in layer_acts if acts]


def train_model(model, n_epochs=5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    train_losses, val_accs = [], []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(imgs), labels)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += labels.size(0)
        val_accs.append(correct / total)
        print(f'  Epoch {epoch+1}: loss={train_losses[-1]:.4f}  val_acc={val_accs[-1]:.4f}')

    return train_losses, val_accs


torch.manual_seed(1)
model_xavier  = DeepMLP('xavier')
model_kaiming = DeepMLP('kaiming')

# Collect init-time layer variances before any training
var_xavier_init  = collect_layer_variances(model_xavier.to(device),  train_loader)
var_kaiming_init = collect_layer_variances(model_kaiming.to(device), train_loader)

print('Xavier init:')
losses_xavier,  accs_xavier  = train_model(model_xavier)
print('\nHe/Kaiming init:')
losses_kaiming, accs_kaiming = train_model(model_kaiming)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training loss
ax = axes[0]
ax.plot(losses_xavier,  'o-', label='Xavier',  linewidth=2)
ax.plot(losses_kaiming, 's-', label='He',      linewidth=2)
ax.set_title('Training Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy')
ax.legend(); ax.grid(True, alpha=0.3)

# Validation accuracy
ax = axes[1]
ax.plot(accs_xavier,  'o-', label='Xavier',  linewidth=2)
ax.plot(accs_kaiming, 's-', label='He',      linewidth=2)
ax.set_title('Validation Accuracy'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)

# Layer-wise variance at initialization
ax = axes[2]
layers = list(range(1, len(var_xavier_init) + 1))
ax.semilogy(layers, var_xavier_init,  'o-', label='Xavier',  linewidth=2)
ax.semilogy(layers, var_kaiming_init, 's-', label='He',      linewidth=2)
ax.set_title('Activation Variance at Init'); ax.set_xlabel('Layer'); ax.set_ylabel('Variance (log)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Xavier final val acc:  {accs_xavier[-1]:.4f}')
print(f'He     final val acc:  {accs_kaiming[-1]:.4f}')

## Section 5 — Orthogonal Initialization for an RNN

Orthogonal hidden-to-hidden weights ensure the recurrent transformation is an isometry — preserving gradient norm across timesteps. We:
1. Verify `W_hh^T @ W_hh ≈ I` numerically
2. Compare gradient norms through time for random vs orthogonal init on a 50-step sequence

In [ ]:
# 1. Verify orthogonality
hidden_size = 128
W_orth  = torch.empty(hidden_size, hidden_size)
W_rand  = torch.empty(hidden_size, hidden_size)

nn.init.orthogonal_(W_orth)
nn.init.normal_(W_rand, std=1.0 / math.sqrt(hidden_size))  # scaled random

I = torch.eye(hidden_size)
print('Orthogonal W_hh:')
print(f'  max|W^T W - I| = {(W_orth.T @ W_orth - I).abs().max():.2e}')
print(f'  max|W W^T - I| = {(W_orth @ W_orth.T - I).abs().max():.2e}')

print('\nScaled random W_hh:')
print(f'  max|W^T W - I| = {(W_rand.T @ W_rand - I).abs().max():.2e}')
print(f'  Singular values: min={W_rand.svd().S.min():.4f}  max={W_rand.svd().S.max():.4f}')
print(f'  (orthogonal: all singular values = 1.0)')

In [ ]:
# 2. Compare gradient norms through BPTT on a 50-step sequence

class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, init='orthogonal'):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ih = nn.Linear(input_size, hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out  = nn.Linear(hidden_size, 1)

        nn.init.xavier_normal_(self.W_ih.weight)
        if init == 'orthogonal':
            nn.init.orthogonal_(self.W_hh.weight)
        else:
            nn.init.normal_(self.W_hh.weight, std=1.0 / math.sqrt(hidden_size))
        nn.init.zeros_(self.W_ih.bias)

    def forward(self, xs):
        """xs: (batch, seq_len, input_size)"""
        h = torch.zeros(xs.size(0), self.hidden_size, device=xs.device)
        hiddens = []
        for t in range(xs.size(1)):
            h = torch.tanh(self.W_ih(xs[:, t, :]) + self.W_hh(h))
            hiddens.append(h)
        return self.out(hiddens[-1]), hiddens


def measure_gradient_norms(model, xs, y):
    """Return per-timestep gradient norms via BPTT."""
    out, hiddens = model(xs)
    loss = F.mse_loss(out, y)
    loss.backward(retain_graph=False)

    # Recompute per-hidden-state gradients w.r.t. the loss
    norms = []
    out, hiddens = model(xs)
    loss = F.mse_loss(out, y)
    for h in hiddens:
        if h.grad_fn is not None:
            grad = torch.autograd.grad(loss, h, retain_graph=True)[0]
            norms.append(grad.norm().item())
    return norms


seq_len = 50
batch   = 32
inp_sz  = 8
hid_sz  = 64

torch.manual_seed(2)
xs = torch.randn(batch, seq_len, inp_sz)
y  = torch.randn(batch, 1)

rnn_orth = VanillaRNN(inp_sz, hid_sz, init='orthogonal')
rnn_rand = VanillaRNN(inp_sz, hid_sz, init='random')

norms_orth = measure_gradient_norms(rnn_orth, xs, y)
norms_rand = measure_gradient_norms(rnn_rand, xs, y)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(norms_orth, 'o-', label='Orthogonal init', linewidth=2, markersize=4)
ax.plot(norms_rand, 's-', label='Scaled random',   linewidth=2, markersize=4)
ax.set_xlabel('Timestep'); ax.set_ylabel('Gradient norm')
ax.set_title('Gradient Norm vs Timestep (linear)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
safe_orth = [max(v, 1e-30) for v in norms_orth]
safe_rand = [max(v, 1e-30) for v in norms_rand]
ax.semilogy(safe_orth, 'o-', label='Orthogonal init', linewidth=2, markersize=4)
ax.semilogy(safe_rand, 's-', label='Scaled random',   linewidth=2, markersize=4)
ax.set_xlabel('Timestep'); ax.set_ylabel('Gradient norm (log scale)')
ax.set_title('Gradient Norm vs Timestep (log)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Orthogonal: first/last grad norm ratio = {norms_orth[0]/max(norms_orth[-1], 1e-30):.2f}')
print(f'Random:     first/last grad norm ratio = {norms_rand[0]/max(norms_rand[-1], 1e-30):.2f}')

## Section 6 — PyTorch Defaults Audit

PyTorch silently initializes weights for you when you instantiate a layer. We inspect and verify the documented defaults for:
`nn.Linear`, `nn.Conv2d`, `nn.Embedding`, `nn.LSTM`, `nn.BatchNorm2d`

In [ ]:
def audit_layer(name, layer, weight_attr='weight', bias_attr='bias'):
    w = getattr(layer, weight_attr, None)
    b = getattr(layer, bias_attr,   None)
    print(f'\n--- {name} ---')
    if w is not None:
        fan_in, fan_out = nn.init._calculate_fan_in_and_fan_out(w)
        bound = 1.0 / math.sqrt(fan_in) if fan_in > 0 else 0
        print(f'  weight shape : {tuple(w.shape)}')
        print(f'  weight mean  : {w.mean().item():.4f}')
        print(f'  weight var   : {w.var().item():.6f}')
        print(f'  expected var : {(2*bound)**2 / 12:.6f}  (Uniform(-1/sqrt(fan_in), 1/sqrt(fan_in)))')
        in_range = (w.abs() <= bound + 1e-6).all().item()
        print(f'  in kaiming_uniform bounds: {in_range}  (bound={bound:.4f})')
    if b is not None:
        print(f'  bias shape   : {tuple(b.shape)}')
        print(f'  bias range   : [{b.min().item():.4f}, {b.max().item():.4f}]')

torch.manual_seed(0)

# nn.Linear: weight ~ Kaiming uniform (fan_in mode, a=sqrt(5))
linear = nn.Linear(512, 256)
audit_layer('nn.Linear(512, 256)', linear)

# nn.Conv2d: same as Linear but fan_in = in_channels * kH * kW
conv = nn.Conv2d(64, 128, kernel_size=3)
audit_layer('nn.Conv2d(64, 128, k=3)', conv)

# nn.Embedding: weight ~ N(0, 1)
emb = nn.Embedding(10000, 128)
print('\n--- nn.Embedding(10000, 128) ---')
print(f'  weight mean: {emb.weight.mean().item():.4f}  (expected ~0)')
print(f'  weight var:  {emb.weight.var().item():.4f}   (expected ~1)')

# nn.LSTM: weight_ih and weight_hh both Kaiming uniform
lstm = nn.LSTM(input_size=32, hidden_size=64, num_layers=1)
print('\n--- nn.LSTM(32, 64) ---')
for attr in ['weight_ih_l0', 'weight_hh_l0', 'bias_ih_l0', 'bias_hh_l0']:
    p = getattr(lstm, attr)
    print(f'  {attr:<20}: shape={tuple(p.shape)}  mean={p.mean().item():.4f}  var={p.var().item():.6f}')

# nn.BatchNorm2d: weight (gamma) = 1, bias (beta) = 0
bn = nn.BatchNorm2d(64)
print('\n--- nn.BatchNorm2d(64) ---')
print(f'  weight (gamma): all ones  = {bn.weight.allclose(torch.ones(64))}')
print(f'  bias   (beta):  all zeros = {bn.bias.allclose(torch.zeros(64))}')

In [ ]:
# Visualise weight distributions at default init
torch.manual_seed(0)
layers_to_plot = {
    'Linear(512,256)':      nn.Linear(512, 256).weight.detach(),
    'Conv2d(64,128,k=3)':   nn.Conv2d(64, 128, 3).weight.detach().flatten(),
    'Embedding(10000,128)': nn.Embedding(10000, 128).weight.detach().flatten(),
    'LSTM(32,64) W_ih':     nn.LSTM(32, 64).weight_ih_l0.detach().flatten(),
}

fig, axes = plt.subplots(1, len(layers_to_plot), figsize=(16, 4))
for ax, (name, weights) in zip(axes, layers_to_plot.items()):
    w = weights.flatten().numpy()
    ax.hist(w, bins=80, density=True, alpha=0.8, color='steelblue')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)

plt.suptitle('Default PyTorch Layer Weight Distributions', y=1.02)
plt.tight_layout(); plt.show()